## Imports

In [0]:
%pip install yfinance

In [0]:
import yfinance as yf
import pandas as pd

## Collect Data

### Get Tickers From Source

In [0]:
def get_tickers(table_name, column_name):
    """
    Get tickers from source

    Args:
        table_name(str): Name of the table containing the tickers.
        column_name(str): Name of the column containing the tickers

    Returns:
        list(str): List of tickers to be used in data collection
    """

    df = spark.table(table_name)

    tickers = (
        df.select(column_name)
        .distinct()
        .dropna()
        .toPandas()[column_name]
        .tolist()
    )

    return tickers

### Collect Prices

In [0]:
def collect_price_data(ticker, period = "2y"):
    """
    Collect historical price data for an asset using Yahoo Finance.

    Args:
        ticker(str): Asset code.
        period(str): Data collection period, default "2y".

    Returns:
        pd.DataFrame: DataFrame containing closing price, volume, ticker and date.
    """
    
    df = yf.Ticker(ticker).history(period=period)

    if df is None or df.empty:
        return pd.DataFrame(columns=["date", "ticker", "close", "volume"])

    df = df.reset_index()

    df = df[["Date", "Close", "Volume"]]

    df.columns = ["date", "close", "volume"]

    df["ticker"] = ticker

    return df

## Build Dataframe

In [0]:
def build_dataset(tickers):
    """
    Build a consolidated dataset from multiple tickers.

    Args
        tickers(list): List of assets to collect.

    Returns
        pd.DataFrame: Consolidated dataset containing all assets
    """

    dfs = []

    for t in tickers:
        df = collect_price_data(t)
        if not df.empty:
            dfs.append(df)

    if not dfs:
        return pd.DataFrame(columns=["date", "ticker", "close", "volume"])

    return pd.concat(dfs, ignore_index=True)

## Convert Dataframe to Spark

In [0]:
def to_spark(df):
    """
    Convert a pandas DataFrame to a Spark DataFrame.

    Args:
        df(DataFrame): Pandas DataFrame.

    Returns:
        pyspark.sql.DataFrame: Spark DataFrame.
    """
    
    df = spark.createDataFrame(df)

    return df

## Save Data

In [0]:
def save_data(df, table_name):
    """
    Save data to a Delta table.

    Args:
        df (DataFrame): Spark DataFrame
        table_name (str): Name of the Delta table

    Returns:
        None
    """

    df.write.format("delta") \
        .mode('overwrite') \
        .option("overwriteSchema", "true") \
        .option("mergeSchema", "true") \
        .saveAsTable(table_name)

## Main Function

In [0]:
def main():
    """
    Runs the data collection and transformation pipeline
    """
    
    tickers = get_tickers("risk_management.funds", "ticker")
    df = build_dataset(tickers)
    spark_df = to_spark(df)
    save_data(spark_df, "risk_management.risk_dataset_bronze")

## Execution

In [0]:
if __name__ == "__main__":
    main()